# Camada Silver — Limpeza e Padronização dos Dados do Olist

Neste notebook lemos as tabelas Bronze e aplicamos todas as transformações exigidas: renomeação de colunas para português, tradução de status, deduplicação com Window Functions, criação de colunas derivadas e junções.

Nenhuma alteração é feita nas tabelas da camada Bronze — todos os dados tratados são salvos em novas tabelas no schema Silver.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [0]:
catalogo = "medalhao"
silver_db = "silver"

In [0]:
customers_df       = spark.table(f"{catalogo}.bronze.tb_customers")
orders_df          = spark.table(f"{catalogo}.bronze.tb_orders")
order_items_df     = spark.table(f"{catalogo}.bronze.tb_order_items")
order_payments_df  = spark.table(f"{catalogo}.bronze.tb_order_payments")
order_reviews_df   = spark.table(f"{catalogo}.bronze.tb_order_reviews")
products_df        = spark.table(f"{catalogo}.bronze.tb_products")
sellers_df         = spark.table(f"{catalogo}.bronze.tb_sellers")
category_transl_df = spark.table(f"{catalogo}.bronze.tb_product_category_name_translation")
cotacao_df         = spark.table(f"{catalogo}.bronze.tb_cotacao_dolar")

print("Tabelas Bronze carregadas com sucesso!")

---
### 1 - silver.dim_consumidores
Origem: `tb_customers`

- Renomear colunas para português
- Deduplicação com Window Function (ROW_NUMBER particionado por id, ordenado por timestamp_ingestion DESC) — mantém apenas o registro mais recente
- Estado e cidade em UPPER CASE

In [0]:
w_customers = Window.partitionBy("customer_id").orderBy(F.col("timestamp_ingestion").desc())

dim_consumidores = (
    customers_df
    .withColumn("rn", F.row_number().over(w_customers))
    .filter(F.col("rn") == 1)
    .select(
        F.col("customer_id").alias("id_consumidor"),
        F.col("customer_unique_id").alias("id_unico_consumidor"),
        F.col("customer_name").alias("nome_consumidor"),
        F.col("customer_gender").alias("genero_consumidor"),
        F.to_date("customer_birth_date").alias("data_nascimento"),
        F.col("customer_age").cast("int").alias("idade_consumidor"),
        F.col("customer_zip_code_prefix").alias("prefixo_cep"),
        F.upper(F.col("customer_city")).alias("cidade"),
        F.upper(F.col("customer_state")).alias("estado")
    )
)

dim_consumidores.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalogo}.{silver_db}.dim_consumidores")

print(f"✅ dim_consumidores: {dim_consumidores.count()} registros")

---
### 2 - silver.fat_pedidos
Origem: `tb_orders`

- Tradução do status de inglês para português
- Colunas derivadas: tempo_entrega_dias, tempo_entrega_estimado_dias, diferenca_entrega_dias, entrega_no_prazo

In [0]:
status_traducao = (
    F.when(F.col("order_status") == "delivered", "entregue")
    .when(F.col("order_status") == "canceled", "cancelado")
    .when(F.col("order_status") == "shipped", "enviado")
    .when(F.col("order_status") == "processing", "processando")
    .when(F.col("order_status") == "invoiced", "faturado")
    .when(F.col("order_status") == "unavailable", "indisponível")
    .when(F.col("order_status") == "created", "criado")
    .when(F.col("order_status") == "approved", "aprovado")
    .otherwise(F.col("order_status"))
)

fat_pedidos = (
    orders_df
    .select(
        F.col("order_id").alias("id_pedido"),
        F.col("customer_id").alias("id_consumidor"),
        status_traducao.alias("status"),
        F.to_timestamp("order_purchase_timestamp").alias("data_pedido"),
        F.to_timestamp("order_approved_at").alias("data_aprovacao"),
        F.to_timestamp("order_delivered_carrier_date").alias("data_envio_transportadora"),
        F.to_timestamp("order_delivered_customer_date").alias("data_entrega_real"),
        F.to_timestamp("order_estimated_delivery_date").alias("data_entrega_estimada")
    )
    .withColumn(
        "tempo_entrega_dias",
        F.datediff(F.col("data_entrega_real"), F.col("data_pedido"))
    )
    .withColumn(
        "tempo_entrega_estimado_dias",
        F.datediff(F.col("data_entrega_estimada"), F.col("data_pedido"))
    )
    .withColumn(
        "diferenca_entrega_dias",
        F.col("tempo_entrega_dias") - F.col("tempo_entrega_estimado_dias")
    )
    .withColumn(
        "entrega_no_prazo",
        F.when(
            (F.col("status") == "entregue") & (F.col("diferenca_entrega_dias") <= 0), "Sim"
        ).when(
            (F.col("status") == "entregue") & (F.col("diferenca_entrega_dias") > 0), "Não"
        ).otherwise("Não Entregue")
    )
)

fat_pedidos.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalogo}.{silver_db}.fat_pedidos")

print(f"✅ fat_pedidos: {fat_pedidos.count()} registros")

---
### 3 - silver.fat_itens_pedidos
Origem: `tb_order_items`

- Renomear colunas conforme mapeamento do enunciado

In [0]:
fat_itens_pedidos = (
    order_items_df
    .select(
        F.col("order_id").alias("id_pedido"),
        F.col("order_item_id").alias("id_item"),
        F.col("product_id").alias("id_produto"),
        F.col("seller_id").alias("id_vendedor"),
        F.col("shipping_limit_date").alias("data_limite_envio"),
        F.col("price").cast("double").alias("preco_BRL"),
        F.col("freight_value").cast("double").alias("preco_frete")
    )
)

fat_itens_pedidos.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalogo}.{silver_db}.fat_itens_pedidos")

print(f"✅ fat_itens_pedidos: {fat_itens_pedidos.count()} registros")

---
### 4 - silver.fat_pagamentos_pedidos
Origem: `tb_order_payments`

- Tradução do tipo de pagamento de inglês para português

In [0]:
tipo_pagamento_pt = (
    F.when(F.col("payment_type") == "credit_card", "Cartão de Crédito")
    .when(F.col("payment_type") == "boleto", "Boleto")
    .when(F.col("payment_type") == "voucher", "Voucher")
    .when(F.col("payment_type") == "debit_card", "Cartão de Débito")
    .when(F.col("payment_type") == "not_defined", "Não Definido")
    .otherwise(F.col("payment_type"))
)

fat_pagamentos_pedidos = (
    order_payments_df
    .select(
        F.col("order_id").alias("id_pedido"),
        F.col("payment_sequential").alias("sequencia_pagamento"),
        tipo_pagamento_pt.alias("tipo_pagamento"),
        F.col("payment_installments").cast("int").alias("parcelas"),
        F.col("payment_value").cast("double").alias("valor_pagamento")
    )
)

fat_pagamentos_pedidos.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalogo}.{silver_db}.fat_pagamentos_pedidos")

print(f"✅ fat_pagamentos_pedidos: {fat_pagamentos_pedidos.count()} registros")

---
### 5 - silver.fat_avaliacoes_pedidos
Origem: `tb_order_reviews`

- Usar `try_to_timestamp` para tolerância a falhas nas colunas de data
- Remover registros com pedido inválido (NULL) ou datas de comentário no futuro
- Preencher títulos e comentários vazios com "Sem título" e "Sem comentário"

In [0]:
# O enunciado mapeia tanto review_comment_title quanto review_comment_message
# para "comentario_avaliacao". Optou-se por separar em titulo_avaliacao e 
# comentario_avaliacao para evitar perda de informação.
fat_avaliacoes_pedidos = (
    order_reviews_df
    .select(
        F.col("review_id").alias("id_avaliacao"),
        F.col("order_id").alias("id_pedido"),
        F.expr("try_cast(review_score as int)").alias("nota_avaliacao"),
        F.coalesce(F.col("review_comment_title"), F.lit("Sem título")).alias("titulo_avaliacao"),
        F.coalesce(F.col("review_comment_message"), F.lit("Sem comentário")).alias("comentario_avaliacao"),
        F.try_to_timestamp(F.col("review_creation_date")).alias("data_criacao_avaliacao"),
        F.try_to_timestamp(F.col("review_answer_timestamp")).alias("data_resposta_avaliacao")
    )
    .filter(F.col("id_pedido").isNotNull())
    .filter(
        (F.col("data_criacao_avaliacao").isNull()) |
        (F.col("data_criacao_avaliacao") <= F.current_timestamp())
    )
)

fat_avaliacoes_pedidos.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalogo}.{silver_db}.fat_avaliacoes_pedidos")

print(f"✅ fat_avaliacoes_pedidos: {fat_avaliacoes_pedidos.count()} registros")

---
### 6 - silver.dim_produtos
Origem: `tb_products`

- Deduplicação sênior com Window Function (mesmo padrão de consumidores)
- Renomear colunas e dimensões para português

In [0]:
w_products = Window.partitionBy("product_id").orderBy(F.col("timestamp_ingestion").desc())

dim_produtos = (
    products_df
    .withColumn("rn", F.row_number().over(w_products))
    .filter(F.col("rn") == 1)
    .select(
        F.col("product_id").alias("id_produto"),
        F.col("product_name").alias("nome_produto"),
        F.coalesce(F.col("product_category_name"), F.lit("sem_categoria")).alias("categoria_produto"),
        F.col("product_weight_g").cast("double").alias("peso_produto_gramas"),
        F.col("product_length_cm").cast("double").alias("comprimento_centimetros"),
        F.col("product_height_cm").cast("double").alias("altura_centimetros"),
        F.col("product_width_cm").cast("double").alias("largura_centimetros"),
        F.col("product_photos_qty").cast("int").alias("quantidade_fotos"),
        F.col("product_name_lenght").cast("int").alias("tamanho_nome_produto"),
        F.col("product_description_lenght").cast("int").alias("tamanho_descricao_produto")
    )
)

dim_produtos.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalogo}.{silver_db}.dim_produtos")

print(f"✅ dim_produtos: {dim_produtos.count()} registros")

---
### 7 - silver.dim_vendedores
Origem: `tb_sellers`

- Deduplicação sênior com Window Function
- Estado e cidade em UPPER CASE

In [0]:
w_sellers = Window.partitionBy("seller_id").orderBy(F.col("timestamp_ingestion").desc())

dim_vendedores = (
    sellers_df
    .withColumn("rn", F.row_number().over(w_sellers))
    .filter(F.col("rn") == 1)
    .select(
        F.col("seller_id").alias("id_vendedor"),
        F.col("seller_name").alias("nome_vendedor"),
        F.to_date("seller_registration_date").alias("data_registro_vendedor"),
        F.col("seller_zip_code_prefix").alias("prefixo_cep"),
        F.upper(F.col("seller_city")).alias("cidade"),
        F.upper(F.col("seller_state")).alias("estado")
    )
)

dim_vendedores.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalogo}.{silver_db}.dim_vendedores")

print(f"✅ dim_vendedores: {dim_vendedores.count()} registros")

---
### 8 - silver.dim_categoria_produtos_traducao
Origem: `tb_product_category_name_translation`

- Renomear colunas para nome_produto_pt e nome_produto_en

In [0]:
dim_categoria_traducao = (
    category_transl_df
    .select(
        F.col("product_category_name").alias("nome_produto_pt"),
        F.col("product_category_name_english").alias("nome_produto_en")
    )
)

categorias_faltantes = spark.createDataFrame([
    ("pc_gamer", "pc_gamer"),
    ("portateis_cozinha_e_preparadores_de_alimentos", "portable_kitchen_and_food_preparers")
], ["nome_produto_pt", "nome_produto_en"])

dim_categoria_traducao = dim_categoria_traducao.unionByName(categorias_faltantes)

dim_categoria_traducao.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalogo}.{silver_db}.dim_categoria_produtos_traducao")

print(f"✅ dim_categoria_produtos_traducao: {dim_categoria_traducao.count()} registros")

---
### 9 - silver.dim_cotacao_dolar
Origem: `tb_cotacao_dolar`

- A API não retorna cotação para finais de semana e feriados
- Criamos um calendário contínuo de datas e preenchemos as lacunas com a cotação de fechamento da sexta-feira anterior usando `last()` com `ignorenulls=True` sobre uma Window Function

In [0]:
cotacao_base = (
    cotacao_df
    .withColumn("data_cotacao", F.to_date(F.col("dataHoraCotacao")))
    .withColumn("cotacao_compra", F.col("cotacaoCompra").cast("double"))
    .select("data_cotacao", "cotacao_compra")
    .groupBy("data_cotacao")
    .agg(F.last("cotacao_compra").alias("cotacao_compra"))
)

datas_range = cotacao_base.agg(
    F.min("data_cotacao").alias("data_min"),
    F.max("data_cotacao").alias("data_max")
).collect()[0]

data_min = datas_range["data_min"]
data_max = datas_range["data_max"]

calendario = spark.sql(f"""
    SELECT explode(sequence(
        to_date('{data_min}'), 
        to_date('{data_max}'), 
        interval 1 day
    )) AS data_cotacao
""")

cotacao_completa = calendario.join(cotacao_base, on="data_cotacao", how="left")

w_cotacao = Window.orderBy("data_cotacao").rowsBetween(Window.unboundedPreceding, Window.currentRow)

dim_cotacao_dolar = (
    cotacao_completa
    .withColumn("cotacao_compra", F.last("cotacao_compra", ignorenulls=True).over(w_cotacao))
)

dim_cotacao_dolar.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalogo}.{silver_db}.dim_cotacao_dolar")

print(f"✅ dim_cotacao_dolar: {dim_cotacao_dolar.count()} registros")

---
### 10 - silver.fat_pedido_total (Tabela Final Silver)
Junção de `fat_pedidos` com `fat_pagamentos_pedidos` e `dim_cotacao_dolar`.

Agrega o valor total pago por pedido em BRL e converte para USD usando a cotação do dia da compra.

Valores financeiros arredondados para 2 casas decimais.

In [0]:
fat_pedidos_silver = spark.table(f"{catalogo}.{silver_db}.fat_pedidos")
fat_pagamentos_silver = spark.table(f"{catalogo}.{silver_db}.fat_pagamentos_pedidos")
dim_cotacao_silver = spark.table(f"{catalogo}.{silver_db}.dim_cotacao_dolar")

pagamentos_agg = (
    fat_pagamentos_silver
    .groupBy("id_pedido")
    .agg(F.sum("valor_pagamento").alias("valor_total_pago_brl"))
)

pedidos_com_pagamento = (
    fat_pedidos_silver
    .join(pagamentos_agg, on="id_pedido", how="inner")
    .withColumn("data_pedido", F.to_date("data_pedido"))
)

fat_pedido_total = (
    pedidos_com_pagamento
    .join(
        dim_cotacao_silver,
        pedidos_com_pagamento["data_pedido"] == dim_cotacao_silver["data_cotacao"],
        how="left"
    )
    .withColumn(
        "valor_total_pago_usd",
        F.round(F.col("valor_total_pago_brl") / F.col("cotacao_compra"), 2)
    )
    .select(
        "id_pedido",
        "id_consumidor",
        "status",
        F.round("valor_total_pago_brl", 2).alias("valor_total_pago_brl"),
        "valor_total_pago_usd",
        "data_pedido"
    )
)

fat_pedido_total.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{catalogo}.{silver_db}.fat_pedido_total")

print(f"✅ fat_pedido_total: {fat_pedido_total.count()} registros")

---
### Otimização Física (Delta Optimize + ZORDER)
Executando OPTIMIZE com ZORDER nas tabelas fato pelas colunas de id_pedido e data_pedido para melhorar a performance de leitura analítica.

In [0]:
%sql

OPTIMIZE medalhao.silver.fat_pedidos ZORDER BY (id_pedido, data_pedido);
OPTIMIZE medalhao.silver.fat_itens_pedidos ZORDER BY (id_pedido);
OPTIMIZE medalhao.silver.fat_pagamentos_pedidos ZORDER BY (id_pedido);
OPTIMIZE medalhao.silver.fat_avaliacoes_pedidos ZORDER BY (id_pedido);
OPTIMIZE medalhao.silver.fat_pedido_total ZORDER BY (id_pedido, data_pedido);

### Validação — Tabelas criadas na camada Silver

In [0]:
%sql

SHOW TABLES IN medalhao.silver;

In [0]:
tabelas_silver = [
    "dim_consumidores", "fat_pedidos", "fat_itens_pedidos",
    "fat_pagamentos_pedidos", "fat_avaliacoes_pedidos",
    "dim_produtos", "dim_vendedores", "dim_categoria_produtos_traducao",
    "dim_cotacao_dolar", "fat_pedido_total"
]

for tabela in tabelas_silver:
    qtd = spark.table(f"{catalogo}.{silver_db}.{tabela}").count()
    print(f"{tabela}: {qtd} registros")